# Математическая база проекта

В этом файле описаны математические модели и формулы, которые лежат в основе проекта. 


## 1. Геометрическое броуновское движение (GBM)

Цена базового актива $S_t$ моделируется с помощью стохастического дифференциального уравнения:

$$
dS_t = \mu S_t dt + \sigma S_t dW_t
$$

Где:
* $\mu$ — ожидаемая доходность (drift).
* $\sigma$ — волатильность.
* $W_t$ — винеровский процесс.

В риск-нейтральной мере доходность заменяется на безрисковую ставку $r$. Тогда точное решение уравнения для конечной цены $S_T$ имеет вид:

$$
S_T = S_0 \exp\left( \left(r - \frac{1}{2}\sigma^2\right)T + \sigma \sqrt{T} Z \right)
$$

Где $Z \sim \mathcal{N}(0, 1)$ — стандартная нормальная случайная величина.

## 2. Аналитическое решение Блэка-Шоулза

Для европейских опционов существует точное аналитическое решение. Сначала вычисляются параметры $d_1$ и $d_2$:

$$
d_1 = \frac{\ln(S_0 / K) + (r + \frac{1}{2}\sigma^2)T}{\sigma \sqrt{T}}
$$

$$
d_2 = d_1 - \sigma \sqrt{T}
$$

Где $\Phi(x)$ — кумулятивная функция распределения стандартного нормального закона (в Python это `norm.cdf()`).

**Цена опциона Call:**
$$
C = S_0 \Phi(d_1) - K e^{-rT} \Phi(d_2)
$$

**Цена опциона Put:**
$$
P = K e^{-rT} \Phi(-d_2) - S_0 \Phi(-d_1)
$$

## 3. Оценка методом Монте-Карло и снижение дисперсии

Математическое ожидание выплаты (payoff) по опциону оценивается как выборочное среднее:

$$
\hat{V} = e^{-rT} \frac{1}{N} \sum_{i=1}^{N} \text{Payoff}(S_T^{(i)})
$$

**Стандартная ошибка (Standard Error, SE):**
$$
SE = \frac{s}{\sqrt{N}}
$$

**Антитетические переменные (Antithetic Variates):**
Оценка строится на усреднении пар $Z$ и $-Z$:
$$
\hat{V}_{anti} = e^{-rT} \frac{1}{N/2} \sum_{i=1}^{N/2} \frac{\text{Payoff}(S_T(Z_i)) + \text{Payoff}(S_T(-Z_i))}{2}
$$

**Контрольные вариаты (Control Variates):**
$$
\hat{V}_{cv} = \hat{V}_{MC} - c^* (X - \mathbb{E}[X])
$$

Где $X$ — контрольная переменная (например, $S_T$), а оптимальный коэффициент $c^*$ равен:
$$
c^* = \frac{\text{Cov}(V, X)}{\text{Var}(X)}
$$

## 5. Паритет опционов Call и Put (Put-Call Parity)

Цены европейских опционов Call и Put жестко связаны между собой. Это следует из отсутствия арбитражных возможностей на рынке. Формула паритета:

$$
C - P = S_0 - K e^{-rT}
$$

Если это равенство нарушается, появляется возможность получить безрисковую прибыль (арбитраж). Именно эта формула проверяется в тесте `test_put_call_parity` в нашем проекте.

## 6. Почему Монте-Карло работает? (Теория вероятностей)

Метод Монте-Карло опирается на два краеугольных камня статистики:

**1. Закон больших чисел (Law of Large Numbers):**
При увеличении количества путей \(N \to \infty\), выборочное среднее \(\hat{V}\) сходится по вероятности к истинному математическому ожиданию \(V\).

**2. Центральная предельная теорема (Central Limit Theorem):**
Ошибка нашего выборочного среднего распределена нормально. Асимптотически:

$$
\sqrt{N} (\hat{V} - V) \xrightarrow{d} \mathcal{N}(0, \sigma^2)
$$

Где \(\sigma^2\) — дисперсия выплаты по опциону. Из этой теоремы напрямую следует, что стандартная ошибка нашего расчета (SE) уменьшается со скоростью:

$$
SE \propto \frac{1}{\sqrt{N}}
$$

Это объясняет, почему на лог-лог графике в `research.ipynb` линия ошибки имеет наклон \(-0.5\).

## 7. Вывод оптимального коэффициента для Control Variates

Мы хотим найти такой коэффициент \(c\), чтобы дисперсия скорректированной оценки была минимальной:

$$
\text{Var}(V_{cv}) = \text{Var}(V - c(X - \mathbb{E}[X]))
$$

Раскрывая дисперсию разности, получаем:

$$
\text{Var}(V_{cv}) = \text{Var}(V) - 2c \cdot \text{Cov}(V, X) + c^2 \text{Var}(X)
$$

Чтобы найти минимум, берем производную по \(c\) и приравниваем к нулю:

$$
\frac{d}{dc} \text{Var}(V_{cv}) = -2\text{Cov}(V, X) + 2c\text{Var}(X) = 0
$$

Отсюда выражаем оптимальный коэффициент:

$$
c^* = \frac{\text{Cov}(V, X)}{\text{Var}(X)}
$$

Именно эта формула реализована в функции `mc_call_control_variate` с использованием `np.cov()`.